![JohnSnowLabs](https://sparknlp.org/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/uncertainty/LLM_Uncertainty_Estimation_in_Spark_NLP.ipynb)

# Estimating LLM Uncertainty in Spark NLP

A complete guide to `LLMUncertaintyEstimator` and the annotators that feed it:
`AutoGGUFModel`, `MPNetEmbeddings`, `SampleEntailmentMatrix`, and `MarsTokenImportance`.

## Motivation

Large language models do not signal uncertainty the way humans do. A model that does not
know the capital of a given country will answer just as fluently, and with the same
apparent confidence, as when it states a well-known fact. There is no built-in "I don't
know" signal in the generated text itself. Any system that acts on an LLM's output - a RAG
answer shown to a user, a label applied to a dataset, an agent step that triggers a side
effect - needs a way to separate reliable answers from unreliable ones based on the model's
behavior rather than its wording.

`LLMUncertaintyEstimator` addresses this by turning one or more sampled completions from an
LLM into a single numeric uncertainty score that can be thresholded, logged, sorted, or
routed on, inside a Spark NLP pipeline at Spark scale.

## Contents

1. The two families of methods (black-box and white-box) and when to use each
2. Runnable code for every method the annotator supports
3. Combining methods with `ensemble`
4. Calibrating a `threshold` to produce a boolean reliability flag
5. Validation results from an end-to-end run
6. Production use cases: RAG gating, human escalation, dataset auditing, model/prompt comparison
7. Known issues and troubleshooting

## Background

`LLMUncertaintyEstimator` implements methods from the uncertainty-estimation literature,
following the original papers' definitions:

- **[Bakman et al., 2025](https://arxiv.org/abs/2506.01114)** benchmarks a wide range of
  uncertainty methods under calibration-set distribution shift. The methods implemented
  here are the ones that retained low error even when the deployment data distribution
  differed from the calibration data - the property that matters most for a production
  uncertainty signal.
- **[Kuhn et al., 2023](https://arxiv.org/abs/2302.09664)** introduces Semantic Entropy:
  cluster sampled answers by meaning rather than exact text match, then compute entropy
  over the resulting clusters. This is the foundation of the black-box method family.
- **MARS** ([Bakman et al.](https://arxiv.org/abs/2402.10999), via
  [TruthTorchLM](https://github.com/Ybakman/TruthTorchLM)) weights each generated token's
  log-probability by its importance to the answer's meaning - for example, "Paris" in "The
  capital is Paris" carries more weight than "is".


## Architecture

`LLMUncertaintyEstimator` computes no logits and loads no model of its own. It is a
post-processor that reads metadata written by upstream annotators. The upstream annotators
required depend on which method or methods are used:



**Black-box methods** (`semanticEntropy`, `eccentricity`) require multiple sampled
completions per prompt, plus a way to determine which samples are semantically equivalent -
either `MPNetEmbeddings` (cosine similarity, the default) or `SampleEntailmentMatrix`
(bidirectional NLI entailment, following the original paper).

**White-box methods** (`meanLogProb`, `perplexity`, `predictiveEntropy`, `mars`) require
per-token log-probabilities from `AutoGGUFModel` and operate on a single sample - no
resampling is needed, so their cost is roughly that of one generation call.

Each of these pipelines is built out in full below.


## Setup

Run the Colab install cell only when using Google Colab. If a GPU is available, keep
`gpu=True`; every annotator in this notebook runs substantially faster with GPU inference,
particularly `AutoGGUFModel` sampling.


In [ ]:
# Only execute this if you are on Google Colab
! wget -q http://setup.johnsnowlabs.com/colab.sh -O - | bash


In [ ]:
import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *
from pyspark.ml import Pipeline
from pyspark.sql.functions import col

spark = sparknlp.start(gpu=True)
print(sparknlp.version())


### Download a GGUF model

This notebook uses `Qwen3-1.7B`: small enough to run quickly in a demonstration
environment, and, as shown in the validation section, capable of producing a useful
uncertainty signal in practice. Any GGUF model can be substituted; the pipelines below are
model-agnostic.

Qwen3 is a reasoning model and wraps its reasoning in `<think>...</think>` before the final
answer by default. This is addressed explicitly in the next section, since it is the most
common cause of incorrect results from the black-box methods.


In [ ]:
EXPORT_PATH = "Qwen3-1.7B-Q4_K_M.gguf"
! wget "https://huggingface.co/unsloth/Qwen3-1.7B-GGUF/resolve/main/Qwen3-1.7B-Q4_K_M.gguf?download=true" -O {EXPORT_PATH}


In [ ]:
document_assembler = DocumentAssembler().setInputCol("text").setOutputCol("document")

# Build the GGUF model ONCE and reconfigure it per section below. Each loadSavedModel call mmaps
# and repacks the full model into native memory that is not necessarily released afterwards, so
# constructing a second instance per method would accumulate off-heap memory for no reason.
# Because it is a single mutable object, every section restates the params it depends on rather
# than relying on what an earlier cell happened to set.
llm = (
    AutoGGUFModel.loadSavedModel(EXPORT_PATH, spark)
    .setInputCols("document")
    .setOutputCol("completions")
    .setBatchSize(5)
    .setNGpuLayers(99)  # set to 0 if you don't have a GPU
)


## Removing the `<think>` block

This is addressed here, before any pipeline is built, because it is the most common cause
of incorrect results in this notebook.

Reasoning models (Qwen3 with thinking enabled, DeepSeek-R1, and similar) prepend a
`<think>...</think>` block to every completion. If left in place and passed to a black-box
method, the boilerplate reasoning phrasing tends to be near-identical across samples even
when the final answers differ substantially. This suppresses the real signal and collapses
semantic clustering into a single spuriously "confident" cluster, producing a low
uncertainty score regardless of whether the model actually agrees with itself.

This is resolved with a single setting:


In [ ]:
llm.setRemoveThinkingTag("think")


This strips `<think>...</think>`, leaving only the final answer for downstream annotators to
consume. Set it for all reasoning models; for non-reasoning models it has no effect.

Two details worth knowing:

- Generation can be cut off by `nPredict` before the closing `</think>` ever appears. In that
  case everything from the unclosed opening tag onwards is stripped, so in-progress reasoning
  never leaks into the answer.
- When `outputLogProbs` is enabled, the `completion_probabilities` array is narrowed to exactly
  the tokens that produced the surviving text. The white-box methods therefore score the answer
  only, and MARS token offsets line up with the stripped answer rather than the raw generation.


## Method 1: Semantic entropy (embeddings backend)

This is the default method. It samples N completions at nonzero temperature, embeds each
one, clusters by cosine similarity, and computes entropy over the resulting cluster sizes.
Five differently-worded but semantically identical answers form one cluster and yield low
entropy (low uncertainty); five meaningfully different answers form several small clusters
and yield high entropy (high uncertainty).

The choice of embedding model matters. `MPNetEmbeddings` is a Sentence-BERT model trained on
NLI/STS-benchmark data for the symmetric task of deciding whether two sentences express the same
meaning, which is exactly what semantic clustering needs - and it is the backend validated here.
Other Sentence-BERT models such as `MiniLMEmbeddings` are trained for the same objective, but note
that they currently drop empty-text inputs instead of embedding them, so a row where one sample
came back empty will fail the estimator's one-embedding-per-sample check. `E5Embeddings` and `BGEEmbeddings` are retrieval-optimized models trained for the
asymmetric task of determining whether a passage answers a query. A retrieval embedder will
run without error here, but it optimizes for a different objective than the one this method
needs.

Cost: N generations per prompt (`setNumSamples(n)`) plus N embedding calls. This is the most
expensive method family in this notebook, and the most extensively validated in the
literature.


In [ ]:
# Restate the sampling params this method needs on the shared `llm` instance.
llm.setNumSamples(5).setTemperature(0.8).setNPredict(150).setOutputLogProbs(False)

embeddings = (
    MPNetEmbeddings.pretrained()
    .setInputCols("completions")
    .setOutputCol("sample_embeddings")
)

uncertainty_semantic_entropy = (
    LLMUncertaintyEstimator()
    .setInputCols(["completions", "sample_embeddings"])
    .setOutputCol("uncertainty")
    .setMethods(["semanticEntropy"])
)

pipeline = Pipeline().setStages(
    [document_assembler, llm, embeddings, uncertainty_semantic_entropy]
)

data = spark.createDataFrame(
    [["What is the capital of France?"], ["What is the capital of Kyrgyzstan?"]],
    ["text"],
)
result = pipeline.fit(data).transform(data)
result.selectExpr(
    "text",
    "uncertainty[0].metadata['semantic_entropy'] as semantic_entropy",
    "uncertainty[0].metadata['num_semantic_clusters'] as num_clusters",
    "uncertainty[0].metadata['uncertainty_score'] as uncertainty_score",
).show(truncate=False)


A well-known fact, such as the capital of France, should produce an entropy near `0.0`,
since every sample converges on the same answer. A genuinely obscure fact should produce a
higher entropy, since temperature-based sampling allows the model's answers to diverge.


## Method 2: Eccentricity

Eccentricity uses the same inputs as semantic entropy (sampled completions and embeddings)
with different underlying math: spectral clustering identifies the consensus answer, then
measures how far each sample sits from that consensus in embedding space. A tight consensus
yields low eccentricity; an outlier sample raises it.

Eccentricity is preferable to semantic entropy when a single divergent sample is more
significant than the overall cluster-size distribution. It reuses the embeddings already
computed for semantic entropy, so it adds no additional cost when both methods are used
together.


In [ ]:
llm.setNumSamples(5).setTemperature(0.8).setNPredict(150).setOutputLogProbs(False)

uncertainty_eccentricity = (
    LLMUncertaintyEstimator()
    .setInputCols(["completions", "sample_embeddings"])
    .setOutputCol("uncertainty")
    .setMethods(["eccentricity"])
    .setEigenThreshold(0.9)
)

pipeline = Pipeline().setStages(
    [document_assembler, llm, embeddings, uncertainty_eccentricity]
)
result = pipeline.fit(data).transform(data)
result.selectExpr("text", "uncertainty[0].metadata['eccentricity'] as eccentricity").show(truncate=False)


## Method 3: Semantic entropy (NLI backend)

The original Semantic Entropy paper (Kuhn et al.) decides whether two samples share a meaning
using bidirectional NLI entailment rather than embedding similarity: sample A and sample B are
equivalent if A entails B and B entails A. `SampleEntailmentMatrix` computes that N x N
entailment matrix. Like `MarsTokenImportance`, it is a plumbing annotator that attaches metadata
without producing a score of its own; `LLMUncertaintyEstimator` reads that metadata when
`similarityBackend` is `"nli"`.

`SampleEntailmentMatrix.pretrained()` loads `bert_base_uncased_mnli_entailment_onnx`, an export of
[textattack/bert-base-uncased-MNLI](https://huggingface.co/textattack/bert-base-uncased-MNLI)
published in this annotator's serialization format.

Note that `pretrained()` only accepts models written by this class. The
`BertForZeroShotClassification` XNLI checkpoints on the Models Hub use a different internal
layout, so they download fine and then fail to deserialize here - use `loadSavedModel` with a
self-exported ONNX checkpoint if you want a different NLI model. See the troubleshooting section
at the end for the label-order caveat that comes with substituting one.


Scoring N samples requires `N*(N-1)` pairwise NLI model calls; the diagonal
(self-entailment) is trivially `1.0` and is skipped. `maxSamplesForNli` (default `10`, up to
90 calls per row) prevents very large batches from being issued silently; raise it
explicitly to cluster more samples with this backend.


In [ ]:
llm.setNumSamples(5).setTemperature(0.8).setNPredict(150).setOutputLogProbs(False)

entailment = (
    SampleEntailmentMatrix.pretrained("bert_base_uncased_mnli_entailment_onnx", "en")
    .setInputCols("completions")
    .setOutputCol("completions_with_entailment")
)

uncertainty_nli = (
    LLMUncertaintyEstimator()
    .setInputCols(["completions_with_entailment"])
    .setOutputCol("uncertainty")
    .setMethods(["semanticEntropy"])
    .setSimilarityBackend("nli")
)

pipeline = Pipeline().setStages(
    [document_assembler, llm, entailment, uncertainty_nli]
)
result = pipeline.fit(data).transform(data)
result.selectExpr("text", "uncertainty[0].metadata['semantic_entropy'] as nli_semantic_entropy").show(truncate=False)


## Method 4: White-box methods (`meanLogProb`, `perplexity`, `predictiveEntropy`)

The black-box methods above ask "does the model agree with itself when resampled?", which costs
N generations per prompt. The white-box family instead reads the per-token log-probabilities the
model already produced during a single generation, so it costs one generation and no resampling.

- `meanLogProb` is the length-normalized mean per-token log-likelihood. Note its direction:
  unlike every other metric here, it is stored confidence-oriented, so values closer to `0` mean
  *more* confident.
- `perplexity` is `exp(-meanLogProb)`, the same signal on a more familiar scale. It is bounded
  below by `1.0` rather than `0.0`, which the `ensemble` normalization accounts for.
- `predictiveEntropy` is the mean per-token entropy over the top-k alternatives at each position,
  so it needs `setNProbs(k > 1)`. Because llama.cpp only returns the top k, this necessarily
  underestimates true predictive entropy - it cannot see mass outside the top k.

All three require `setOutputLogProbs(True)`.


In [ ]:
# White-box methods read per-token logprobs from a single generation, so numSamples goes back
# to 1 and outputLogProbs is switched on.
llm.setNumSamples(1).setOutputLogProbs(True).setNProbs(5)  # nProbs > 1 is required by predictiveEntropy

uncertainty_whitebox = (
    LLMUncertaintyEstimator()
    .setInputCols(["completions"])
    .setOutputCol("uncertainty")
    .setMethods(["meanLogProb", "perplexity", "predictiveEntropy"])
)

pipeline = Pipeline().setStages(
    [document_assembler, llm, uncertainty_whitebox]
)
result = pipeline.fit(data).transform(data)
result.selectExpr(
    "text",
    "uncertainty[0].metadata['mean_log_prob'] as mean_log_prob",
    "uncertainty[0].metadata['perplexity'] as perplexity",
    "uncertainty[0].metadata['predictive_entropy'] as predictive_entropy",
).show(truncate=False)


## Method 5: MARS (token-importance-weighted white-box score)

Plain `meanLogProb` weights every token equally. In "The capital of France is Paris", though, the
model's confidence about "Paris" is far more informative than its confidence about "is" or "the".
MARS ([Bakman et al.](https://arxiv.org/abs/2402.11756)) weights each token's log-probability by a
learned importance score before averaging, so the result is dominated by the model's confidence in
the parts of the answer that carry meaning.

Like `SampleEntailmentMatrix`, `MarsTokenImportance` is a plumbing annotator. It takes two
`DOCUMENT` input columns in this order - the question, then the sampled answer(s) - and attaches
`token_importance` metadata that `LLMUncertaintyEstimator` reads and joins against the answer's
per-token log-probabilities by character offset.

`MarsTokenImportance.pretrained()` loads `mars_token_importance`, an export of the
[duygunuryldz/MARS](https://huggingface.co/duygunuryldz/MARS) checkpoint (a
`BertForTokenClassification` with `num_labels=3`).


In [ ]:
question_assembler = DocumentAssembler().setInputCol("text").setOutputCol("question")

llm.setNumSamples(1).setOutputLogProbs(True).setNProbs(5)

mars = (
    MarsTokenImportance.pretrained("mars_token_importance", "en")
    .setInputCols(["question", "completions"])
    .setOutputCol("completions_with_mars")
)

uncertainty_mars = (
    LLMUncertaintyEstimator()
    .setInputCols(["completions_with_mars"])
    .setOutputCol("uncertainty")
    .setMethods(["mars"])
)

pipeline = Pipeline().setStages(
    [document_assembler, question_assembler, llm, mars, uncertainty_mars]
)
result = pipeline.fit(data).transform(data)
result.selectExpr("text", "uncertainty[0].metadata['mars'] as mars").show(truncate=False)


## Combining methods with `ensemble`

No single method is optimal for every situation: semantic entropy captures whether the
model agrees with itself across resamples, while MARS captures the model's confidence in
the parts of the answer that carry meaning. Setting `ensemble=True` with multiple methods
produces a combined, normalized score in addition to each method's individual raw score.


In [ ]:
# Combine a black-box and a white-box signal. Both must be computable from the same row, so this
# pipeline samples 5 completions AND records logprobs.
llm.setNumSamples(5).setTemperature(0.8).setNPredict(150).setOutputLogProbs(True).setNProbs(5)

uncertainty_ensemble = (
    LLMUncertaintyEstimator()
    .setInputCols(["completions", "sample_embeddings"])
    .setOutputCol("uncertainty")
    .setMethods(["semanticEntropy", "perplexity"])
    .setEnsemble(True)
    # Positional: one weight per method, in the same order. Omit to weight them equally.
    .setEnsembleWeights([2.0, 1.0])
)

pipeline = Pipeline().setStages(
    [document_assembler, llm, embeddings, uncertainty_ensemble]
)
result = pipeline.fit(data).transform(data)
result.selectExpr(
    "text",
    "uncertainty[0].metadata['semantic_entropy'] as semantic_entropy",
    "uncertainty[0].metadata['perplexity'] as perplexity",
    "uncertainty[0].metadata['uncertainty_score'] as ensembled_score",
    "uncertainty[0].metadata['confidence_score'] as confidence_score",
).show(truncate=False)


Each method keeps its own raw score in the metadata; `uncertainty_score` is the combined value.
Ensembling normalizes per method first (`semanticEntropy` by its exact `ln(numSamples)` maximum,
`eccentricity` by a `sqrt(numSamples)` scale heuristic, and the white-box methods not at all,
since they have no natural bound without calibration), so a combined score is approximate until
you calibrate a threshold on your own data.


## Calibration: converting a raw score into a reliability decision

A raw uncertainty score does not by itself indicate whether an answer should be trusted; a
value of `0.42` has no inherent meaning without context. The literature this annotator is
based on found that decision thresholds must be calibrated on data resembling the target
deployment distribution, or error rates rise sharply. This step should not be skipped, and
a threshold calibrated for a different use case should not be reused.

### Calibration workflow

1. Collect a labeled set of `(question, model_answer, is_correct)` triples representative
   of the target use case - ideally several dozen examples spanning the model's actual
   knowledge boundary rather than a randomly selected development set (see the note on
   question curation in the validation section below).
2. Run the selected method(s) over that set and record the uncertainty scores.
3. Select a threshold that achieves the desired false-positive/false-negative tradeoff, for
   example via an ROC-curve sweep.
4. Apply it:


In [ ]:
llm.setNumSamples(5).setTemperature(0.8).setNPredict(150).setOutputLogProbs(False)

uncertainty_with_threshold = (
    LLMUncertaintyEstimator()
    .setInputCols(["completions", "sample_embeddings"])
    .setOutputCol("uncertainty")
    .setMethods(["semanticEntropy"])
    .setThreshold(0.5)  # replace with YOUR calibrated value
)

pipeline = Pipeline().setStages(
    [document_assembler, llm, embeddings, uncertainty_with_threshold]
)
# .cache() matters here: `gated_result` is reused by several cells below (RAG gating, human
# escalation, the safety gate). Spark DataFrames are lazy, so without caching each of those cells
# would independently re-trigger this pipeline - including fresh LLM sampling at temperature 0.8 -
# and could see different is_reliable values than the ones printed here.
gated_result = pipeline.fit(data).transform(data).cache()
gated_result.selectExpr(
    "text",
    "uncertainty[0].metadata['uncertainty_score'] as uncertainty_score",
    "uncertainty[0].metadata['is_reliable'] as is_reliable",
).show(truncate=False)


`is_reliable` appears in the metadata only once `threshold` is set. Without a threshold,
only the raw score is returned, which is the correct default: an unset, silently-incorrect
boolean would be worse than no boolean at all.


## Validation results

This section reports results from an end-to-end run: 25 QA pairs mixing well-known facts
with genuinely obscure ones, real `Qwen3-1.7B` generations, and every method computed
directly, scored by AUROC (how well the uncertainty score separates incorrect answers from
correct ones; `0.5` is random guessing and `1.0` is perfect separation):

| Method | AUROC |
|---|---|
| `mars` | **0.774** |
| `semanticEntropy` (NLI backend) | **0.773** |
| `ensemble` | 0.750 |
| `meanLogProb` | 0.738 |
| `perplexity` | 0.738 |
| `predictiveEntropy` | 0.738 |
| `semanticEntropy` (embeddings backend) | 0.708 |
| `eccentricity` | 0.702 |

Every method exceeds random guessing by a wide margin, clustered in the 0.70-0.77 range
across all eight independently-computed methods. `mars` and the NLI-backend
`semanticEntropy`, the two methods most faithful to the original literature, achieved the
highest scores.

### Limitations

- **Sample size.** With n=25 and only 3-4 incorrect answers, the confidence interval on any
  individual AUROC value is wide. These results should be treated as directional evidence
  that the mechanism works, not as a calibrated, generalizable accuracy guarantee, which is
  the reason the calibration step described above is necessary.
- **Question curation.** Several questions selected as obscure (for example, the capitals of
  Kyrgyzstan and Burkina Faso) were in fact answered correctly and confidently by this
  model, since modern LLMs have been trained on a large volume of world-fact text.
  Calibrating for a specific use case requires probing the target model's actual knowledge
  boundary rather than assuming which questions are difficult.
- **Per-example noise.** In one run, a correctly-answered question about the Treaty of
  Tordesillas received a higher uncertainty score than the single incorrect answer in the
  set. The aggregate signal (AUROC across many examples) is reliable; any individual score
  can still be noisy.


## Production use cases

The following sections describe five common ways to integrate `LLMUncertaintyEstimator`
into a production Spark NLP pipeline.

### 1. RAG answer gating

The most common use case: run the RAG pipeline as usual, then check `is_reliable` before
displaying the answer to a user. When `is_reliable` is `false`, fall back to a
lower-confidence response, such as the retrieved passages themselves, rather than
presenting a fabricated-sounding answer.


In [ ]:
from pyspark.sql.functions import when, expr

FALLBACK = "I found some relevant information but am not confident in a direct answer."

gated = gated_result.withColumn(
    "user_facing_answer",
    when(
        expr("uncertainty[0].metadata['is_reliable'] = 'true'"),
        col("completions")[0]["result"],
    ).otherwise(expr(f"'{FALLBACK}'")),
)
gated.selectExpr("text", "user_facing_answer").show(truncate=False)


### 2. Human-in-the-loop escalation

Route only the uncertain fraction of traffic to a human reviewer, rather than reviewing all
traffic (which does not scale) or none (which allows confident errors through unchecked).
Since white-box methods require only a single generation, this is inexpensive enough to run
on every response rather than a sample.


In [ ]:
needs_human_review = gated_result.filter(
    expr("uncertainty[0].metadata['is_reliable'] = 'false'")
)
auto_resolved = gated_result.filter(
    expr("uncertainty[0].metadata['is_reliable'] = 'true'")
)
print(f"escalate: {needs_human_review.count()}, auto-resolve: {auto_resolved.count()}")
# needs_human_review.write... -> your ticketing/review queue


### 3. Auditing an existing dataset

For datasets of labels, summaries, or answers already generated by an LLM in bulk, the
white-box methods can be run over the existing generations (single sample, low cost) to
identify likely errors without manual review of the full dataset. Sort by
`uncertainty_score` in descending order and review the highest-scoring examples first. This
is the lowest-cost way to apply this annotator, since it requires no resampling and works on
data for which log-probabilities are already available.


In [ ]:
# Sort an already-scored DataFrame by uncertainty and review the top N first - this is where a
# fixed labeling budget has the highest expected value. In a real audit this DataFrame would come
# from re-running your existing generations through a white-box pipeline (single sample, so it
# costs one generation per row and no resampling).
audit_queue = (
    gated_result
    .selectExpr("text", "uncertainty[0].metadata['uncertainty_score'] as uncertainty_score")
    .orderBy(col("uncertainty_score").desc())
)
audit_queue.show(20, truncate=False)


### 4. Comparing prompts or models by reliability

When A/B testing two prompts or models, accuracy on a small evaluation set can be noisy and
requires ground-truth labels. Average uncertainty score requires no labels: run both
variants over the same question set and compare the resulting distributions. A prompt that
reduces average uncertainty without a corresponding reduction in accuracy represents a
genuine improvement in self-consistency, and typically correlates with improved real-world
reliability.


In [ ]:
def average_uncertainty(scored_df):
    """Mean uncertainty over a scored DataFrame - no ground-truth labels needed."""
    return scored_df.selectExpr(
        "avg(cast(uncertainty[0].metadata['uncertainty_score'] as double)) as avg_uncertainty"
    ).collect()[0]["avg_uncertainty"]


# Run the same question set through each variant's pipeline, then compare. Here both sides use
# the one pipeline built above; in a real A/B you would build two, differing in prompt or model.
print("variant A:", average_uncertainty(gated_result))
# print("variant B:", average_uncertainty(variant_b_result))


### 5. Safety gating before agent actions

When an LLM's output triggers a consequential action, such as a database write, an email
send, or a tool call, execution can be gated on `is_reliable`. This is an ideal use case for
the white-box method family: single-sample, low-latency, and inexpensive enough to run
inline on every agent step rather than on a sample of traffic.


In [ ]:
safe_to_execute = gated_result.filter(expr("uncertainty[0].metadata['is_reliable'] = 'true'"))
needs_confirmation = gated_result.filter(expr("uncertainty[0].metadata['is_reliable'] = 'false'"))
# safe_to_execute    -> proceed automatically
# needs_confirmation -> ask the user to confirm, or reject the action
print(f"auto-execute: {safe_to_execute.count()}, confirm first: {needs_confirmation.count()}")


## Cost comparison

| Method | Samples required | Requires `outputLogProbs` | Additional annotator | Relative cost |
|---|---|---|---|---|
| `meanLogProb` / `perplexity` | 1 | Yes | None | Low (one generation) |
| `predictiveEntropy` | 1 | Yes (+ `setNProbs(k>1)`) | None | Low |
| `mars` | 1 | Yes | `MarsTokenImportance` | Moderate (one BERT call per sampled answer) |
| `semanticEntropy` (embeddings) | N (e.g., 5) | No | `MPNetEmbeddings` | High (N generations + N embeddings) |
| `eccentricity` | N | No | `MPNetEmbeddings` | High (reuses embeddings above) |
| `semanticEntropy` (NLI) | N | No | `SampleEntailmentMatrix` | Highest (`N*(N-1)` pairwise forward passes, batched) |

For cost- or latency-sensitive deployments, such as inline production gating at high query
volume, the white-box family is recommended. For offline auditing or use cases with budget
for resampling, the black-box family - particularly the NLI backend - provides the
strongest, most literature-faithful signal.


## Known issues and troubleshooting

1. **Reasoning models require `setRemoveThinkingTag("think")`.** Covered above and repeated here
   because it is the most common cause of support issues: if black-box scores are uniformly low
   regardless of the question, this setting is almost certainly the cause.

2. **Do not reconstruct `AutoGGUFModel` inside a loop.** Build a single instance (via
   `pretrained()` or `loadSavedModel`) and reuse it across every `.fit()`/`.transform()` call.
   Each construction mmaps and repacks the full GGUF model into native (off-heap) memory that is
   not necessarily released when the wrapper goes out of scope. Reloading per row or per request
   accumulates native memory and can terminate the JVM well before heap usage looks like it is
   under pressure. This notebook builds one instance and reconfigures it per section - note that
   this makes `llm` a single mutable object, which is why each section restates the params it
   depends on instead of relying on what an earlier cell set.

3. **`pretrained()` only accepts models written by the same annotator class.**
   `SampleEntailmentMatrix` reads an ONNX file under `sample_entailment_matrix_onnx` and
   `MarsTokenImportance` one under `mars_token_importance_onnx`. A model published for a
   different annotator - for instance the `BertForZeroShotClassification` XNLI checkpoints, which
   look like a reasonable NLI substitute - downloads fine and then fails to deserialize. Use
   `bert_base_uncased_mnli_entailment_onnx` and `mars_token_importance`, or `loadSavedModel` with your
   own ONNX export.

4. **NLI checkpoint label order is model-specific.** Do not assume the textbook GLUE MNLI
   convention (`entailment=0, neutral=1, contradiction=2`); many checkpoints ship no `id2label`
   mapping at all. `textattack/bert-base-uncased-MNLI`, for instance, has only `LABEL_0/1/2`
   placeholders, and its actual trained order - confirmed empirically against unambiguous probe
   sentences - is `contradiction, entailment, neutral`. Confirm the order for any checkpoint you
   substitute, the same way. The annotator resolves entailment by *name*, so `labels.txt` being
   wrong silently gives you a different class's probability.

5. **`meanLogProb` uses an inverted convention relative to the other metrics.** Every other
   metadata value follows "higher means more uncertain". `mean_log_prob` is stored in its natural,
   confidence-oriented sense (closer to `0` means more confident, as a log-probability should
   read) and is sign-flipped only internally when folded into `uncertainty_score`. Account for
   this when reading `mean_log_prob` directly.

6. **Use a Sentence-BERT model, not a retrieval embedder, for the embeddings backend.** See
   Method 1. Note that `MPNetEmbeddings` is the backend validated here: it embeds empty
   completions rather than dropping them, which is what keeps one embedding per sampled
   completion. An embedder that drops empty inputs will fail the estimator's 1:1 count check on
   any row where a sample came back empty.

7. **Cost scales quickly with the black-box methods.** `numSamples(n)` requires `n` full
   generations per row, and the NLI backend adds `n*(n-1)` pairwise forward passes on top (padded
   and run `batchSize` at a time, but still quadratic in `n`). Consult the cost table above
   before defaulting to the most expensive option.

8. **`ensembleWeights` is positional.** There must be exactly one weight per entry in `methods`,
   in the same order, all non-negative and not all zero. Mismatches are rejected when the
   pipeline starts rather than partway through a job.


## Reference pipeline

The following combines every method above into a single pipeline that computes all of them
in one pass. A single sampling call feeds all downstream annotators, since
`MPNetEmbeddings`, `MarsTokenImportance`, and the white-box metadata all read from the same
`completions` column.


In [ ]:
# One sampling call feeds every downstream annotator: MPNetEmbeddings, MarsTokenImportance and
# the white-box metadata all read the same `completions` column.
llm.setNumSamples(5).setTemperature(0.8).setNPredict(150).setBatchSize(5)
llm.setRemoveThinkingTag("think").setOutputLogProbs(True).setNProbs(5)

embeddings_full = (
    MPNetEmbeddings.pretrained()
    .setInputCols("completions")
    .setOutputCol("sample_embeddings")
)

mars_full = (
    MarsTokenImportance.pretrained("mars_token_importance", "en")
    .setInputCols(["question", "completions"])
    .setOutputCol("completions_with_mars")
)

uncertainty_full = (
    LLMUncertaintyEstimator()
    .setInputCols(["completions_with_mars", "sample_embeddings"])
    .setOutputCol("uncertainty")
    .setMethods(
        ["semanticEntropy", "eccentricity", "meanLogProb", "perplexity", "predictiveEntropy", "mars"]
    )
    .setEnsemble(True)
    .setThreshold(0.5)  # replace with your own calibrated value
)

full_pipeline = Pipeline().setStages(
    [document_assembler, question_assembler, llm, embeddings_full, mars_full, uncertainty_full]
)

full_data = spark.createDataFrame(
    [
        ["What is the capital of France?"],
        ["What is the smallest country in Africa by area?"],
    ],
    ["text"],
)
full_result = full_pipeline.fit(full_data).transform(full_data)

full_result.selectExpr(
    "text",
    "uncertainty[0].metadata['semantic_entropy'] as semantic_entropy",
    "uncertainty[0].metadata['eccentricity'] as eccentricity",
    "uncertainty[0].metadata['mean_log_prob'] as mean_log_prob",
    "uncertainty[0].metadata['perplexity'] as perplexity",
    "uncertainty[0].metadata['predictive_entropy'] as predictive_entropy",
    "uncertainty[0].metadata['mars'] as mars",
    "uncertainty[0].metadata['uncertainty_score'] as ensemble_uncertainty_score",
    "uncertainty[0].metadata['is_reliable'] as is_reliable",
).show(truncate=False, vertical=True)
